# Codebase Test Notebook

Tests every function in the codebase in isolation so you can inspect inputs, outputs, and catch regressions.

**Sections that require AI2-THOR** are clearly marked — skip them if the simulator is not running.

| Section | Needs AI2-THOR? |
|---------|----------------|
| 1. PolicyLSTM | No |
| 2. PPOAgent | No |
| 3. RewardConfig | No |
| 4. ThorEnv internals (mocked) | No |
| 5. ThorEnv with live simulator | **Yes** |
| 6. Full mini-episode | **Yes** |

In [ ]:
import sys
from pathlib import Path

# Make project root importable regardless of where notebook is launched from
root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent
sys.path.insert(0, str(root))

import math
import numpy as np
import torch
from unittest.mock import MagicMock, patch

print(f'Project root: {root}')
print(f'torch: {torch.__version__}')

---
## 1. PolicyLSTM

Input = flat vector: `vis_features | GPS(3) | compass(2) | action_onehot(n_actions) | res_level(1) | budget(1)`  
Output = `(action_logits, value, hidden)`

### 1.1  Input dimension calculation

In [ ]:
from src.models.lstm import PolicyLSTM

VIS_DIM   = 8
N_ACTIONS = 10
HIDDEN_DIM = 32

model = PolicyLSTM(vis_dim=VIS_DIM, n_actions=N_ACTIONS, hidden_dim=HIDDEN_DIM)

expected = VIS_DIM + 3 + 2 + N_ACTIONS + 1 + 1   # = 25
actual   = model.lstm.input_size
assert actual == expected, f'input_dim mismatch: got {actual}, expected {expected}'

INPUT_DIM = actual
print(f'input_dim = {actual}')
print(f'  breakdown: {VIS_DIM} vis + 3 GPS + 2 compass + {N_ACTIONS} action + 1 res + 1 budget')
print(f'lstm hidden_dim  = {model.lstm.hidden_size}')
print(f'policy_head      = Linear({model.lstm.hidden_size}, {N_ACTIONS})')
print(f'value_head       = Linear({model.lstm.hidden_size}, 1)')

### 1.2  Single-step forward — output shapes

In [ ]:
obs = torch.randn(1, INPUT_DIM)   # (batch=1, input_dim)
logits, value, hidden = model(obs)

assert logits.shape == (1, N_ACTIONS), f'logits: expected (1,{N_ACTIONS}), got {logits.shape}'
assert value.shape  == (1, 1),         f'value: expected (1,1), got {value.shape}'
assert isinstance(hidden, tuple) and len(hidden) == 2
assert hidden[0].shape == (1, 1, HIDDEN_DIM)   # (num_layers, batch, hidden)
assert hidden[1].shape == (1, 1, HIDDEN_DIM)

print(f'logits shape : {logits.shape}')
print(f'value  shape : {value.shape}')
print(f'hidden h_n   : {hidden[0].shape}  (num_layers, batch, hidden_dim)')
print(f'hidden c_n   : {hidden[1].shape}')
print()
probs = torch.softmax(logits, dim=-1).detach().squeeze()
print(f'action probs : {probs.numpy().round(3)}')
print(f'value        : {value.item():.4f}')

### 1.3  Sequence input (T > 1) — only last timestep is returned

In [ ]:
T = 5
obs_seq = torch.randn(1, T, INPUT_DIM)
logits_seq, value_seq, hidden_seq = model(obs_seq)

assert logits_seq.shape == (1, N_ACTIONS)
assert value_seq.shape  == (1, 1)

print(f'Sequence input shape : {obs_seq.shape}')
print(f'logits output shape  : {logits_seq.shape}  <- last timestep only')

### 1.4  Hidden state — passing it changes the output

In [ ]:
obs_a = torch.randn(1, INPUT_DIM)
obs_b = torch.randn(1, INPUT_DIM)

_, _, hidden_after_a    = model(obs_a, hidden=None)
logits_with_hist, _, _  = model(obs_b, hidden=hidden_after_a)
logits_no_hist,   _, _  = model(obs_b, hidden=None)

diff = (logits_with_hist - logits_no_hist).abs().max().item()
assert diff > 0, 'LSTM output must differ when hidden state is provided'

print(f'logits with history  : {logits_with_hist.detach().numpy().round(4)}')
print(f'logits no history    : {logits_no_hist.detach().numpy().round(4)}')
print(f'max absolute diff    : {diff:.6f}  (> 0 confirms hidden state has effect)')

---
## 2. PPOAgent

Interface: `act()` → `store()` each step → `update()` at episode end.

### 2.1  Buffer starts empty

In [ ]:
from src.agents.ppo_agent import PPOAgent

policy = PolicyLSTM(vis_dim=VIS_DIM, n_actions=N_ACTIONS, hidden_dim=HIDDEN_DIM)
agent  = PPOAgent(policy=policy, lr=1e-3, epochs=2)

assert len(agent.buffer) == 0
print(f'buffer length on init: {len(agent.buffer)}')

### 2.2  act() — output types and shapes

In [ ]:
obs = torch.randn(1, INPUT_DIM)
action_idx, log_prob, value, hidden = agent.act(obs)

assert isinstance(action_idx, int),                   f'action_idx must be int, got {type(action_idx)}'
assert 0 <= action_idx < N_ACTIONS,                   f'action_idx={action_idx} out of range'
assert log_prob.shape == torch.Size([]),               f'log_prob must be scalar, got {log_prob.shape}'
assert log_prob.item() <= 0,                           f'log_prob must be ≤ 0 (it is a log probability)'
assert value.shape == torch.Size([1]),                 f'value must be shape (1,), got {value.shape}'
assert isinstance(hidden, tuple) and len(hidden) == 2

print(f'action_idx : {action_idx}  (int in [0, {N_ACTIONS}))')
print(f'log_prob   : {log_prob.item():.4f}  (scalar ≤ 0)')
print(f'value      : {value.item():.4f}')
print(f'hidden     : h_n {hidden[0].shape}, c_n {hidden[1].shape}')

### 2.3  store() — accumulates transitions in buffer

In [ ]:
N_STEPS = 6
obs_flat = obs.squeeze(0)

for i in range(N_STEPS):
    done = (i == N_STEPS - 1)
    agent.store(obs_flat, action_idx, log_prob, reward=float(i) * 0.1, value=value.squeeze(), done=done)

assert len(agent.buffer) == N_STEPS

expected_keys = {'obs', 'action', 'log_prob', 'reward', 'value', 'done'}
assert set(agent.buffer[0].keys()) == expected_keys

print(f'buffer length after {N_STEPS} stores : {len(agent.buffer)}')
print(f'buffer entry keys : {set(agent.buffer[0].keys())}')
print()
for i, t in enumerate(agent.buffer):
    print(f'  step {i}: reward={t["reward"]:.2f}  done={t["done"]}')

### 2.4  update() — clears buffer and returns loss dict

In [ ]:
losses = agent.update()

assert len(agent.buffer) == 0,                          'buffer must be empty after update()'
assert set(losses.keys()) == {'policy_loss', 'value_loss', 'entropy'}

print(f'buffer after update()  : {len(agent.buffer)}  (cleared)')
print(f'loss keys              : {set(losses.keys())}')
print(f'  policy_loss  : {losses["policy_loss"]:.4f}')
print(f'  value_loss   : {losses["value_loss"]:.4f}')
print(f'  entropy      : {losses["entropy"]:.4f}  (should be > 0 for a stochastic policy)')

### 2.5  _compute_returns() — verify against known values

In [ ]:
agent05 = PPOAgent(policy=policy, gamma=0.5, epochs=1)

# Case 1: single step, not done
# R = reward + gamma * last_value = 1.0 + 0.5 * 2.0 = 2.0
r1 = agent05._compute_returns([1.0], [False], torch.tensor(2.0))
assert abs(r1[0].item() - 2.0) < 1e-5, f'Expected 2.0, got {r1[0].item()}'
print(f'Single step (r=1.0, not done, last_value=2.0) → return = {r1[0].item():.4f}  [expected 2.0]')

# Case 2: single step, done → last_value is ignored
# R = reward + gamma * 0 = 1.0
r2 = agent05._compute_returns([1.0], [True], torch.tensor(99.0))
assert abs(r2[0].item() - 1.0) < 1e-5, f'Expected 1.0, got {r2[0].item()}'
print(f'Single step (r=1.0, done, last_value=99) → return = {r2[0].item():.4f}  [expected 1.0, last_value ignored]')

# Case 3: two steps, second is done
# R[-1] = 1.0  (done)
# R[-2] = 1.0 + 0.5 * 1.0 = 1.5
r3 = agent05._compute_returns([1.0, 1.0], [False, True], torch.tensor(99.0))
assert abs(r3[1].item() - 1.0) < 1e-5,  f'Last return should be 1.0, got {r3[1].item()}'
assert abs(r3[0].item() - 1.5) < 1e-5,  f'First return should be 1.5, got {r3[0].item()}'
print(f'Two steps (both r=1.0, second done) → returns = {r3.tolist()}  [expected [1.5, 1.0]]')

---
## 3. RewardConfig

Dataclass that holds all reward shaping scalars.

In [ ]:
from src.simulation.thor_env import RewardConfig

cfg = RewardConfig()

defaults = {
    'step_penalty':        0.002,
    'sense_penalty':       0.02,
    'oversensing_penalty': 0.05,
    'bump_penalty':        0.03,
    'fail_penalty':        1.0,
    'success_reward':      5.0,
    'distance_scale':      0.01,
}
for field, expected in defaults.items():
    actual = getattr(cfg, field)
    assert actual == expected, f'{field}: expected {expected}, got {actual}'
    print(f'  {field:25s} = {actual}')

print()
cfg2 = RewardConfig(success_reward=10.0, fail_penalty=2.0)
assert cfg2.success_reward == 10.0
assert cfg2.fail_penalty   == 2.0
assert cfg2.step_penalty   == 0.002   # other fields unchanged
print(f'Custom config: success_reward={cfg2.success_reward}, fail_penalty={cfg2.fail_penalty}')

---
## 4. ThorEnv internals — mocked (no AI2-THOR needed)

We patch the AI2-THOR `Controller` so `ThorEnv` can be instantiated without a running simulator,
then inject a mock frame + metadata to test internal methods.

### 4.1  Setup — instantiate with mocked controller

In [ ]:
from src.simulation.thor_env import ThorEnv, RewardConfig

with patch('src.simulation.thor_env.Controller'):
    env = ThorEnv(
        base_resolution=(64, 64),
        max_steps=10,
        max_sensing_budget=5,
        seed=42,
        reward_cfg=RewardConfig(),
    )

# Inject a mock frame + metadata
mock_event = MagicMock()
mock_event.frame = np.random.randint(0, 255, (64, 64, 3), dtype=np.uint8)
mock_event.metadata = {
    'agent': {'position': {'x': 0.0, 'y': 0.0, 'z': 0.0}},
    'objects': [],
    'lastActionSuccess': True,
}
env.current_event  = mock_event
env.target_obj_type = 'Chair'

print(f'base_downgrade      : {env.base_downgrade}  (floor(log2(64)) = {math.floor(math.log2(64))})')
print(f'max_steps           : {env.max_steps}')
print(f'max_sensing_budget  : {env.max_sensing_budget}')
print(f'action_list         : {env.action_list}')

### 4.2  base_downgrade for common resolutions

In [ ]:
print('Resolution  base_downgrade  worst block size')
print('─' * 45)
for res in [(64,64), (128,128), (224,224), (256,256)]:
    bd = math.floor(math.log2(min(res)))
    print(f'  {res}      {bd}               {2**bd}×{2**bd} pixels')

### 4.3  _compute_obs() — resolution downgrading

In [ ]:
print('downgrade  block_size  unique_values_ch0  obs_shape')
print('─' * 52)
for k in range(0, env.base_downgrade + 1):
    env._current_downgrade = k
    obs = env._compute_obs()

    assert obs.shape == (3, 64, 64), f'Unexpected shape {obs.shape}'
    assert obs.min() >= 0.0 and obs.max() <= 1.0, 'Values must be in [0, 1]'

    unique = obs[0].unique().numel()
    print(f'  k={k}        2^{k}={2**k:3d}       {unique:4d}               {obs.shape}')

print()
print('Unique values drop as k increases (more blurring = fewer distinct pixel values)')

### 4.4  SENSE timing — improvement is deferred one step

In [ ]:
env._current_downgrade       = 4
env._remaining_sensing_budget = 5
env._step_count               = 0

print(f'Before SENSE: _current_downgrade = {env._current_downgrade}')

# Replicate what step() does for SENSE
env._current_action = 'SENSE'
env._step_count    += 1

# 1) Set validity flag
env._last_sense_was_valid = env._current_downgrade > 0 and env._remaining_sensing_budget > 0

# 2) Compute obs at OLD downgrade
obs_this_step = env._compute_obs()
downgrade_at_obs_time = env._current_downgrade

# 3) THEN decrement
if env._last_sense_was_valid:
    env._current_downgrade      -= 1
    env._remaining_sensing_budget -= 1

print(f'Downgrade when obs was computed : {downgrade_at_obs_time}  <- still old value')
print(f'Downgrade after SENSE step      : {env._current_downgrade}  <- will apply NEXT step')
print(f'_last_sense_was_valid           : {env._last_sense_was_valid}')
print()
assert downgrade_at_obs_time == 4, 'Obs must be computed at OLD downgrade'
assert env._current_downgrade == 3, 'Downgrade must decrease after step'
print('SENSE timing correct: agent pays penalty now, sees improvement next step')

### 4.5  _fail_checker()

In [ ]:
env.max_steps = 10

for count, expected in [(9, False), (10, True), (11, True)]:
    env._step_count = count
    result = env._fail_checker()
    assert result == expected, f'step_count={count}: expected {expected}, got {result}'
    print(f'  step_count={count:2d}, max_steps={env.max_steps} → fail_checker={result}')

### 4.6  _get_min_distance_to_object() — including empty-scene guard

In [ ]:
env.current_event.metadata['agent']['position'] = {'x': 0.0, 'y': 0.0, 'z': 0.0}
env.current_event.metadata['objects'] = [
    {'objectType': 'Chair', 'position': {'x': 3.0, 'y': 0.0, 'z': 4.0}},   # dist = 5.0
    {'objectType': 'Chair', 'position': {'x': 1.0, 'y': 0.0, 'z': 0.0}},   # dist = 1.0
    {'objectType': 'Table', 'position': {'x': 0.0, 'y': 0.0, 'z': 2.0}},   # dist = 2.0
]

d_chair   = env._get_min_distance_to_object('Chair')
d_table   = env._get_min_distance_to_object('Table')
d_missing = env._get_min_distance_to_object('Sofa')    # not in scene

assert abs(d_chair - 1.0) < 1e-5,  f'min Chair dist: expected 1.0, got {d_chair}'
assert abs(d_table - 2.0) < 1e-5,  f'Table dist: expected 2.0, got {d_table}'
assert d_missing == np.inf,          f'missing type: expected inf, got {d_missing}'

print(f'Min dist to Chair (at 1.0 and 5.0) : {d_chair:.2f}')
print(f'Dist to Table (at 2.0)             : {d_table:.2f}')
print(f'Dist to missing type Sofa          : {d_missing}  <- guard returns inf, no crash')

### 4.7  _check_success()

In [ ]:
env.target_obj_type   = 'Chair'
env.success_distance  = 1.5
env.current_event.metadata['agent']['position'] = {'x': 0.0, 'y': 0.0, 'z': 0.0}

scenarios = [
    # (objects list,                                              expected, label)
    (
        [{'objectType': 'Chair', 'position': {'x': 1.0, 'y': 0.0, 'z': 0.0}, 'visible': True}],
        True, 'visible Chair at dist=1.0 (limit=1.5) → success'
    ),
    (
        [{'objectType': 'Chair', 'position': {'x': 1.0, 'y': 0.0, 'z': 0.0}, 'visible': False}],
        False, 'Chair close but NOT visible → fail'
    ),
    (
        [{'objectType': 'Chair', 'position': {'x': 5.0, 'y': 0.0, 'z': 0.0}, 'visible': True}],
        False, 'Chair visible but dist=5.0 > limit=1.5 → fail'
    ),
    (
        [
            {'objectType': 'Chair', 'position': {'x': 5.0, 'y': 0.0, 'z': 0.0}, 'visible': True},
            {'objectType': 'Chair', 'position': {'x': 1.0, 'y': 0.0, 'z': 0.0}, 'visible': True},
        ],
        True, 'Two Chairs: one far, one close → success (any match)'
    ),
]

for objects, expected, label in scenarios:
    env.current_event.metadata['objects'] = objects
    result = env._check_success()
    assert result == expected, f'FAIL: {label} — expected {expected}, got {result}'
    print(f'  {result!s:5}  {label}')

### 4.8  _compute_reward() — one scenario per action type

In [ ]:
cfg = env.cfg   # RewardConfig with defaults

# Fix closest_distance == current_distance so no progress shaping fires
env.target_obj_type    = 'Chair'
env.current_event.metadata['agent']['position'] = {'x': 0.0, 'y': 0.0, 'z': 0.0}
env.current_event.metadata['objects'] = [
    {'objectType': 'Chair', 'position': {'x': 3.0, 'y': 0.0, 'z': 4.0}, 'visible': False},
]
env._closest_distance = env._get_min_distance_to_object('Chair')   # 5.0

def check_reward(action, last_success, valid_sense, truncated, expected, label):
    env._current_action     = action
    env._last_sense_was_valid = valid_sense
    env.current_event.metadata['lastActionSuccess'] = last_success
    r = env._compute_reward(truncated)
    ok = abs(r - expected) < 1e-5
    print(f'  {"OK" if ok else "FAIL":4}  {label:50s}  got {r:+.4f}  expected {expected:+.4f}')
    assert ok

check_reward('MoveAhead', last_success=True,  valid_sense=True,  truncated=False, expected=-cfg.step_penalty,                              label='move success')
check_reward('MoveAhead', last_success=False, valid_sense=True,  truncated=False, expected=-cfg.bump_penalty,                             label='move blocked (bump)')
check_reward('SENSE',     last_success=True,  valid_sense=True,  truncated=False, expected=-cfg.sense_penalty,                            label='valid SENSE')
check_reward('SENSE',     last_success=True,  valid_sense=False, truncated=False, expected=-(cfg.sense_penalty+cfg.oversensing_penalty),  label='invalid SENSE (budget=0)')
check_reward('MoveAhead', last_success=True,  valid_sense=True,  truncated=True,  expected=-cfg.step_penalty - cfg.fail_penalty,          label='move + truncated (timeout)')

---
## 5. ThorEnv with live AI2-THOR simulator

> **Requires a running AI2-THOR simulator.**  
> On a headless server set `platform=CloudRendering` in `ThorEnv.__init__`  
> or ensure a virtual display is available (Xvfb on Linux).

In [ ]:
try:
    live_env = ThorEnv(
        base_resolution=(224, 224),
        max_steps=20,
        max_sensing_budget=5,
        seed=42,
    )
    print('ThorEnv initialised')
    print(f'  base_downgrade : {live_env.base_downgrade}')
    THOR_AVAILABLE = True
except Exception as e:
    print(f'AI2-THOR not available: {e}')
    THOR_AVAILABLE = False

In [ ]:
if THOR_AVAILABLE:
    obs = live_env.reset('FloorPlan1')

    assert obs.shape == (3, 224, 224), f'obs shape: {obs.shape}'
    assert obs.min() >= 0.0 and obs.max() <= 1.0
    assert live_env._current_downgrade == live_env.base_downgrade
    assert live_env._remaining_sensing_budget == live_env.max_sensing_budget
    assert live_env.target_obj_type is not None

    print(f'reset() OK')
    print(f'  obs shape            : {obs.shape}')
    print(f'  obs range            : [{obs.min():.3f}, {obs.max():.3f}]')
    print(f'  target_obj_type      : {live_env.target_obj_type}')
    print(f'  _current_downgrade   : {live_env._current_downgrade}  (= base_downgrade, worst res)')
    print(f'  _remaining_budget    : {live_env._remaining_sensing_budget}')

In [ ]:
if THOR_AVAILABLE:
    sense_id = live_env.action2id['SENSE']
    pre_dg   = live_env._current_downgrade
    pre_bg   = live_env._remaining_sensing_budget

    obs_s, rew_s, term, trunc, info = live_env.step(sense_id)

    assert not term and not trunc
    assert abs(rew_s - (-live_env.cfg.sense_penalty)) < 1e-5,  f'sense reward wrong: {rew_s}'
    assert live_env._current_downgrade == pre_dg - 1,           'downgrade must decrease after SENSE'
    assert live_env._remaining_sensing_budget == pre_bg - 1,    'budget must decrease after SENSE'

    print(f'SENSE step')
    print(f'  reward               : {rew_s:.4f}  (expected -{live_env.cfg.sense_penalty})')
    print(f'  downgrade before     : {pre_dg}  →  after : {live_env._current_downgrade}  (deferred)')
    print(f'  budget before        : {pre_bg}  →  after : {live_env._remaining_sensing_budget}')
    print(f'  _last_sense_was_valid: {live_env._last_sense_was_valid}')

In [ ]:
if THOR_AVAILABLE:
    move_id = live_env.action2id['MoveAhead']

    obs_m, rew_m, term, trunc, info = live_env.step(move_id)

    assert live_env._current_downgrade == live_env.base_downgrade, \
        f'Move must reset downgrade to base. Got {live_env._current_downgrade}'

    print(f'MoveAhead step')
    print(f'  reward               : {rew_m:.4f}')
    print(f'  _current_downgrade   : {live_env._current_downgrade}  (reset to base_downgrade={live_env.base_downgrade})')

In [ ]:
if THOR_AVAILABLE:
    done_id = live_env.action2id['DONE']
    obs_d, rew_d, term, trunc, info = live_env.step(done_id)

    assert term == True,  'DONE must set terminated=True'
    assert trunc == False

    print(f'DONE step')
    print(f'  terminated : {term}')
    print(f'  success    : {info["success"]}')
    print(f'  reward     : {rew_d:.4f}  (success={info["success"]})')
    expected_done_reward = live_env.cfg.success_reward if info['success'] else -live_env.cfg.fail_penalty
    print(f'  expected   : {expected_done_reward:.4f}')
    assert abs(rew_d - expected_done_reward) < 1e-5

In [ ]:
if THOR_AVAILABLE:
    # Exhaust sensing budget and verify oversensing penalty
    live_env.reset('FloorPlan1')
    sense_id = live_env.action2id['SENSE']

    print('Exhausting sensing budget step by step:')
    for i in range(live_env.max_sensing_budget + 2):   # go 2 over budget
        dg_before = live_env._current_downgrade
        bg_before = live_env._remaining_sensing_budget
        _, rew, _, _, _ = live_env.step(sense_id)
        valid = live_env._last_sense_was_valid
        print(f'  sense {i+1}: budget {bg_before}→{live_env._remaining_sensing_budget}  '
              f'dg {dg_before}→{live_env._current_downgrade}  '
              f'valid={valid}  reward={rew:.4f}')

In [ ]:
if THOR_AVAILABLE:
    live_env.close()
    print('env closed')

---
## 6. Full mini-episode — end-to-end

Runs a complete episode with the real agent loop and verifies:
- `agent.buffer` is populated step by step
- `agent.update()` clears the buffer and returns losses
- `info['success']` tracks actual navigation success

In [ ]:
if THOR_AVAILABLE:
    from src.agents.ppo_agent import PPOAgent
    from src.models.lstm import PolicyLSTM

    ep_env = ThorEnv(base_resolution=(224, 224), max_steps=15, seed=0)
    flat_dim = 3 * 224 * 224
    ep_policy = PolicyLSTM(vis_dim=flat_dim, n_actions=len(ep_env.action_list))
    ep_agent  = PPOAgent(policy=ep_policy)

    obs    = ep_env.reset('FloorPlan1')
    obs    = obs.flatten().unsqueeze(0)
    hidden = None
    done   = False
    total_reward = 0.0

    print(f'Target: {ep_env.target_obj_type}')
    print(f'{'step':>4}  {'action':>12}  {'reward':>8}  {'dg':>3}  {'budget':>6}  {'buffer':>6}')
    print('─' * 55)

    while not done:
        action_idx, log_prob, value, hidden = ep_agent.act(obs, hidden)
        next_obs, reward, terminated, truncated, info = ep_env.step(action_idx)
        next_obs = next_obs.flatten().unsqueeze(0)

        ep_agent.store(obs.squeeze(0), action_idx, log_prob, reward, value.squeeze(), terminated or truncated)

        obs = next_obs
        total_reward += reward
        done = terminated or truncated

        print(f'{info["step"]:>4}  {ep_env.action_list[action_idx]:>12}  '
              f'{reward:>+8.4f}  {info["downgrade"]:>3}  {info["sensing_budget"]:>6}  {len(ep_agent.buffer):>6}')

    print('─' * 55)
    assert len(ep_agent.buffer) == info['step'], \
        f'buffer length {len(ep_agent.buffer)} != steps {info["step"]}'

    losses = ep_agent.update()

    assert len(ep_agent.buffer) == 0, 'buffer must be empty after update()'

    print(f'\nEpisode done')
    print(f'  steps        : {info["step"]}')
    print(f'  total reward : {total_reward:.4f}')
    print(f'  success      : {info["success"]}')
    print(f'  buffer       : {len(ep_agent.buffer)} (cleared)')
    print(f'  policy_loss  : {losses["policy_loss"]:.4f}')
    print(f'  value_loss   : {losses["value_loss"]:.4f}')
    print(f'  entropy      : {losses["entropy"]:.4f}')

    ep_env.close()
else:
    print('Skipped — AI2-THOR not available')